# Behavior Modeling API: InterSim Implementation

## Introduction

Tactics2D provides unified reimplementations of a collection of representative traffic participant behavior models to support the development, validation, and testing of Autonomous Driving Systems (ADS) with realistic and scalable traffic interactions. InterSim is one of the default behavior models integrated into Tactics2D.

Original paper: [InterSim: Interactive Traffic Simulation via Explicit Relation Modeling](https://arxiv.org/abs/2210.14431)

Original code: [Tsinghua-MARS-Lab/InterSim](https://github.com/Tsinghua-MARS-Lab/InterSim)

InterSim models interaction through **directed relations**. A pair of vehicles that share a conflict point is reduced to a discrete, directed relation `[influencer -> reactor]`: the influencer passes the conflict point first and the reactor has to yield. A rule-based EnvPlanner then turns those relations into speed profiles, so the interaction structure is explicit rather than hidden inside a learned policy. The original repository pairs this with a learned relation predictor and a marginal trajectory predictor; in Tactics2D the relation direction and the planner are reimplemented with geometry and rules, which keeps the behavior model free of checkpoints and external runtimes.

The original paper validates InterSim on the Waymo Open Motion Dataset (WOMD) interactive validation split. Rather than reproducing only these benchmark settings, Tactics2D reimplements InterSim on top of its unified data representation, map abstraction, and scenario interface. Consequently, the behavior model can be executed on all datasets supported by Tactics2D, making InterSim one of the framework's default foundational behavior models and providing a consistent behavior generation pipeline across heterogeneous traffic datasets.

In this documentation, we will demonstrate how to use the InterSim behavior model in Tactics2D with all built-in modules and datasets.

## Environment Setup

Please install Tactics2D (`pip install 'tactics2[behavior]'`) or add the Tactics2D source directory to your `PYTHONPATH`. See the [Installation Guide](https://tactics2d.readthedocs.io/en/latest/installation/) for more details.

## Dataset Preparation

Tactics2D does not require datasets to be stored in a fixed location. You can place a dataset in any directory and provide its path when loading it. The examples below use paths relative to this folder (`docs/tutorial/`) - `../../../data/...` for the datasets and `../../data/...` for the maps that ship with the repository. **Adjust them to match your local data layout.** Throughout this tutorial, **WOMD** is used to demonstrate the standard workflow of InterSim, while **InD** serves as an additional example to show how InterSim can be used with other datasets supported by Tactics2D.

## Use InterSim for Behavior Generation

The pipeline below demonstrates a complete InterSim closed-loop visualization. Every step that interacts with data (parsing, representing, planning, and rendering) is handled by Tactics2D's public API. The notebook only provides glue code.

| Module | Key API |
|--------|---------|
| **Dataset parsers** | `parser.parse_trajectory(...)` → `(participants, time_range)` |
| **Map abstraction** | `map_.lanes`, `map_.roadlines`, `map_.areas` |
| **Participant model** | `participant.trajectory.get_state(frame)` |
| **Behavior model** | `model.plan(participants, map_, frame, agent_ids)` → `InterSimPlanResult` |
| **Closed loop** | `model.run_closed_loop(participants, map_, ego_id)` → metrics + poses |
| **BEVCamera** | `camera.update(frame, participants, ...)` → `geometry_data` |
| **MatplotlibRenderer** | `renderer.update(geometry_data)` |
| **Trajectory gradient** | `renderer.draw_gradient_trace(positions, colormap)` — BuPu for the driven path, GnBu for the current plan |
| **Color & style** | `participant.color = "light-pink"` overrides `COLOR_PALETTE` defaults |

In [ ]:
%matplotlib notebook

import warnings

warnings.filterwarnings("ignore")

import logging
from pathlib import Path

logging.basicConfig(level=logging.WARNING)

import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from shapely.geometry import Point

from tactics2d.behavior import InterSimBehaviorModel, InterSimConfig
from tactics2d.dataset_parser import LevelXParser, WOMDParser
from tactics2d.display.renderers import MatplotlibRenderer
from tactics2d.display.sensor import BEVCamera
from tactics2d.map.map_config import IND_MAP_CONFIG
from tactics2d.map.parser import OSMParser
from tactics2d.participant.element import Vehicle
from tactics2d.participant.trajectory import State, Trajectory

In [ ]:
mpl.rcParams.update(
    {
        "figure.dpi": 200,
        "font.family": "DejaVu Sans Mono",
        "font.size": 8,
        "animation.html": "html5",
        "animation.embed_limit": 100 * 1024 * 1024,
        "axes.edgecolor": "black",
        "axes.linewidth": 0.8,
        "axes.facecolor": "white",
        "figure.facecolor": "white",
    }
)

In [ ]:
# InterSim model config (the reproduction defaults; the learned predictors stay closed)
INTERSIM_CFG = InterSimConfig(cruise_speed=8.0)

# Prediction / display constants
WARMUP_MS = INTERSIM_CFG.planning_warmup_steps * INTERSIM_CFG.step_ms
EGO_COLOR = "light-pink"
PERCEPTION_RANGE = 200  # BEVCamera perception range (meters)
VIEW_WIDTH = 120  # local view width centered on ego (meters)
VIEW_HEIGHT = 80  # local view height centered on ego (meters)
PLAYBACK_STEP = 1  # render every N-th simulated step (1 = real time at 10 fps)

print("relation mode:", INTERSIM_CFG.relation_mode)
print("step time:", INTERSIM_CFG.step_ms, "ms")
print("planning horizon:", INTERSIM_CFG.horizon_steps, "steps =", INTERSIM_CFG.horizon_steps * INTERSIM_CFG.dt, "s")
print("interaction distance:", INTERSIM_CFG.interaction_distance, "m")
print("closed-loop cadence: warmup", INTERSIM_CFG.planning_warmup_steps,
      "frames, replan every", INTERSIM_CFG.planning_interval, "frames")

## Closed-Loop Usage

The following four steps demonstrate a reusable pipeline for visualizing InterSim closed-loop behavior. Each step describes the relevant built-in modules provided by Tactics2D and how they are combined in the example code.

InterSim is not a take-over model. Instead of replacing one vehicle's future with a prediction, the runner replays a whole scenario: the ego and the vehicles whose predicted futures conflict with it are re-planned at every planning frame, while every other vehicle keeps its ground-truth motion. The visualization therefore re-animates the **simulated poses** returned by the runner, not the parsed ground truth.

### Step 1: Select The Ego Vehicle

`Vehicle` carries a `Trajectory` (a `dict[int, State]` mapping timestamps to positions) and a `color` attribute consumed by the renderer. The ego is the vehicle the closed loop re-plans, so `choose_ego` prefers a vehicle that is already present before the warm-up frame and that survives into the second half of the scenario - the closest local proxy for the WOMD "to-predict" role, and the same heuristic the reproduction harness uses. Pass `ego_id` to `run_interactive_scenario` to override this.

WOMD tracks are not contiguous, so the helper only considers vehicles that actually hold a state at the planning frame. `frame_at_or_after` returns the first frame at or after the warm-up, which is the frame the directed relations are read from.

In [ ]:
def choose_ego(participants, warmup_ms=WARMUP_MS):
    """Pick a persistent early vehicle as the ego (the to-predict proxy)."""
    candidates = []
    for agent_id, participant in participants.items():
        if not isinstance(participant, Vehicle):
            continue
        first_frame = participant.trajectory.first_frame
        last_frame = participant.trajectory.last_frame
        if first_frame is None or last_frame is None:
            continue
        candidates.append(
            (agent_id, first_frame, last_frame, len(participant.trajectory.frames))
        )
    if not candidates:
        raise RuntimeError("The scenario contains no vehicle with a trajectory.")

    last_frame = max(item[2] for item in candidates)
    first_frame = min(item[1] for item in candidates)
    warmup = first_frame + warmup_ms
    late = first_frame + 0.75 * (last_frame - first_frame)
    persistent = [item for item in candidates if item[1] <= warmup and item[2] >= late]
    pool = persistent or candidates
    return min(pool, key=lambda item: (item[1], -item[3], str(item[0])))[0]


def frame_at_or_after(participant, frame_ms):
    """Return the first frame of the trajectory at or after ``frame_ms``."""
    frames = participant.trajectory.frames
    later = [f for f in frames if f >= frame_ms]
    return later[0] if later else frames[-1]

### Step 2: Predict Directed Relations with InterSim

`InterSimBehaviorModel` exposes the interaction structure through a single call shared by all Tactics2D behavior models:

```python
predicted = model.predict(participants, map_, frame=frame, agent_ids=[ego_id])
```

`predict()` returns `{agent_id: Trajectory}` and drops the relations. This tutorial calls the richer sibling instead, which returns every planned trajectory plus the relation graph and the per-agent action:

```python
result = model.plan(participants, map_, frame=frame, agent_ids=[ego_id])
```

- `participants` — the same dict from `parse_trajectory()`; each `Vehicle`'s history states up to `frame` are the model's input.
- `map_` — the `Map` from `parse_map()`, providing the lane centerlines the reference paths follow.
- `agent_ids` — which participants to plan for. The remaining vehicles within `interaction_distance` are still modelled, because they can influence the requested ones.

The result carries four fields:

| Field | Meaning |
|-------|---------|
| `relations` | `Edge(influencer, reactor, reactor_step, frame_diff)` records — "the influencer passes the conflict point first, the reactor must yield" |
| `actions` | `"yield"` or `"follow"` per modelled agent; `"yield"` means the conflict resolver had to brake that agent |
| `trajectories` | the planned future states, reachable with the same `.get_state(frame)` API as history |
| `scene_agent_ids` | the agents this call actually modelled |

Every internally modelled vehicle first rolls a lane-following (or constant-velocity) baseline, overlapping baselines are turned into directed relations, and every reactor decelerates so the final forecast is collision-free. Note the frame convention: `frame` is in **milliseconds**, and the returned trajectories start one step *after* the current frame, so the first planned state is `frame + step_ms`.

#### Where the planned path comes from

InterSim does not predict trajectories; it **rolls agents along the map**. Every modelled agent goes through the same three steps:

1. **Lane matching.** `match_lane` finds the lane whose centerline the agent's current pose is closest to, within `lane_match_radius` (5 m) and `lane_heading_tolerance_deg` (20°). The pose is snapped onto that centerline at arc length `s0`.
2. **Lane chaining.** `lane_chain_points` walks from the matched lane through its successors until the chain covers `lookahead = v0 * horizon_steps * dt + length + stop_margin + 10` metres — i.e. as far as the agent could travel over the horizon, plus a margin. At a fork the next lane is chosen by heading continuity; when the agent carries a `goal_xy`, the successor whose initial heading best aims at that goal wins instead.
3. **Speed profile rollout.** The agent advances along that polyline by a speed profile (`arcs_from_speeds` → `ArcPath.sample_poses`). Conflict resolution only edits the profile, never the path.

When lane matching fails (an agent off the map, or a map without usable centerlines) the reference path degrades to a straight line and the agent keeps a constant velocity.

The closed loop sets `goal_xy` for every agent to its **last ground-truth position in the scenario**, so the chains aim at where each vehicle actually ends up. That is why the drawn plans follow the real lane geometry through junctions and look like routes rather than straight-line extrapolations — they are lane centerline chains, and the goal only breaks ties at forks.

`plan_interaction_scene` below wraps the two calls used by the visualization: the first discovers which vehicles belong to the ego's scene, the second returns the planned trajectories for that whole set — the same ego-first-then-relevant-set ordering the closed loop uses.

In [ ]:
def plan_interaction_scene(model, participants, map_, frame, ego_id):
    """Plan the ego's interaction scene (ego plus the vehicles it conflicts with)."""
    probe = model.plan(participants, map_, frame, agent_ids=[ego_id])
    scene_ids = [agent_id for agent_id in probe.scene_agent_ids if agent_id in participants]
    return model.plan(participants, map_, frame, agent_ids=scene_ids)

### Step 3: Visualize The Directed Relations

The figure below draws one planning frame in the ego's neighbourhood. Grey lines are the map lanes, dotted lines are the ground-truth history the vehicles have already driven, thick lines are the planned trajectories, and each label shows the planned action of that agent.

Red arrows are the directed relations: the tail sits on the influencer and the head points at the reactor, so the arrow reads "this vehicle influences that one, and that one yields".

Only the relations that involve the **ego** are drawn. A dense WOMD scenario can produce dozens of edges among neighbouring vehicles, which turns the figure into arrow spaghetti without telling you anything about the ego — scenario 9 below has 73 edges in total and only 2 that touch the ego. The full graph is still in `result.relations` and is printed by `run_interactive_scenario`.

An arrow is always drawn even when its influencer sits outside the view or is not a vehicle at all; the arrow is clipped by the axes and only the agents inside the view get a label. A relation against a pedestrian or a cyclist has no planned trajectory to draw, but it is usually the relation that actually makes the ego brake.

In [ ]:
def planned_xy(trajectory):
    states = [trajectory.get_state(frame) for frame in trajectory.frames]
    return np.asarray([[state.x, state.y] for state in states])


def history_xy(participant, frame):
    frames = sorted(f for f in participant.trajectory.history_states if f <= frame)
    if not frames:
        return np.zeros((0, 2))
    return np.asarray([participant.trajectory.get_state(f).location for f in frames])


def pose_at(participants, agent_id, frame):
    """Position of an agent, or None when its track has no state at that frame."""
    if agent_id not in participants:
        return None
    if not participants[agent_id].trajectory.has_state(frame):
        return None
    return np.asarray(participants[agent_id].trajectory.get_state(frame).location)


def plot_directed_relations(
    participants, map_, result, ego_id, frame, radius=30.0, title=None, save_to=None
):
    ego_current = pose_at(participants, ego_id, frame)

    def inside_window(point):
        return (
            abs(point[0] - ego_current[0]) <= radius and abs(point[1] - ego_current[1]) <= radius
        )

    def in_neighbourhood(agent_id):
        pose = pose_at(participants, agent_id, frame)
        return pose is not None and inside_window(pose)

    def clipped_to_window(points):
        mask = np.array([inside_window(point) for point in points])
        return np.where(mask[:, None], points, np.nan)

    # A relation may involve a pedestrian or a cyclist, which is exactly the case
    # where a vehicle has to yield; those agents have no planned trajectory in
    # ``result.trajectories`` but the relation still belongs in the figure.
    window_relations = [
        edge for edge in result.relations if ego_id in (edge.influencer, edge.reactor)
    ]

    # Only the agents inside the view are labelled; the arrow of a relation whose
    # influencer sits further away is still drawn and simply clipped by the axes.
    related_ids = {
        agent_id
        for edge in window_relations
        for agent_id in (edge.influencer, edge.reactor)
        if in_neighbourhood(agent_id)
    }
    related_ids.add(ego_id)
    related_ids = sorted(related_ids, key=str)
    color_by_agent = {aid: plt.cm.tab10(index % 10) for index, aid in enumerate(related_ids)}

    figure, axes = plt.subplots(figsize=(7.5, 7.5))
    for lane in map_.lanes.values():
        for side in (lane.left_side, lane.right_side):
            if side is None or side.is_empty:
                continue
            xy = np.asarray(side.coords)
            axes.plot(xy[:, 0], xy[:, 1], color="#d4d4d4", linewidth=0.7, alpha=0.8, zorder=1)

    for agent_id in related_ids:
        color = color_by_agent[agent_id]
        history = clipped_to_window(history_xy(participants[agent_id], frame))
        if len(history) >= 2:
            axes.plot(history[:, 0], history[:, 1], color=color, linewidth=1.0,
                      linestyle=":", alpha=0.9, zorder=5)
        if agent_id in result.trajectories:
            planned = clipped_to_window(planned_xy(result.trajectories[agent_id]))
            axes.plot(planned[:, 0], planned[:, 1], color=color,
                      linewidth=3.4 if agent_id == ego_id else 1.8,
                      alpha=1.0 if agent_id == ego_id else 0.75, zorder=8)
        state = pose_at(participants, agent_id, frame)
        axes.plot(state[0], state[1], marker="o", color=color, markersize=4, zorder=15)
        action = result.actions.get(agent_id, "?")
        label = ("%s (ego): %s" % (agent_id, action)) if agent_id == ego_id else (
            "%s: %s" % (agent_id, action)
        )
        axes.annotate(label, xy=(state[0], state[1]), xytext=(0, 11), textcoords="offset points",
                      color=color, fontsize=8, fontweight="bold", ha="center",
                      bbox={"facecolor": "white", "edgecolor": color, "alpha": 0.9, "pad": 1.2},
                      zorder=30)

    for edge in window_relations:
        start = pose_at(participants, edge.influencer, frame)
        end = pose_at(participants, edge.reactor, frame)
        if start is None or end is None:
            continue
        direction = end - start
        length = float(np.linalg.norm(direction))
        if length < 1e-3:
            continue
        unit = direction / length
        axes.annotate("", xy=end - unit * 3.0, xytext=start + unit * 3.0,
                      arrowprops={"arrowstyle": "-|>", "color": "#d62728", "linewidth": 2.0,
                                  "shrinkA": 0, "shrinkB": 0}, zorder=20)

    axes.plot(ego_current[0], ego_current[1], marker="o", color="black", markersize=6, zorder=25)
    axes.set_xlim(ego_current[0] - radius, ego_current[0] + radius)
    axes.set_ylim(ego_current[1] - radius, ego_current[1] + radius)
    axes.set_aspect("equal")
    axes.set_title(
        title
        or "InterSim directed relations at frame %s ms (%d ego-centric edges of %d)"
        % (frame, len(window_relations), len(result.relations)),
        fontsize=10,
    )
    figure.tight_layout()
    if save_to is not None:
        Path(save_to).parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(save_to, dpi=140)
        print("  saved:", save_to)
    return figure

### Step 4: Render with BEVCamera and MatplotlibRenderer

**BEVCamera** handles viewport culling, coordinate transforms, and geometry generation. `camera.update(...)` returns `(geometry_data, road_set, participant_set)` — the renderer consumes `geometry_data` directly.

**MatplotlibRenderer** draws the scene with automatic z-ordering (road areas < lines < buildings < vehicles < point clouds) and resolves colors from `COLOR_PALETTE` and `DEFAULT_COLOR` by element type.

**Trajectory gradient** (`renderer.enable_trajectory_gradient()` + `draw_gradient_trace()`) renders colormapped past/future traces at zorder 5 — near segments get the dark end, far segments the light end.

The runner does not mutate the input participants - it returns per-agent pose arrays instead, where each row is `[x, y, z, yaw]` and `-1` marks a frame in which the agent is not present. `rebuild_render_participants` therefore rebuilds renderable participants from those poses, holding the last valid pose across gaps, and the ego is coloured through the participant attribute.

The `update()` closure below is called by `FuncAnimation` on every frame: clear old traces → collect active participants → get ego pose → `camera.update()` → `renderer.update()` → draw the driven trace (BuPu, reversed so the closest point is first) → draw the current plan (GnBu).

Two traces are drawn, the same pairing the BITS and LimSim demos use. The **BuPu** trace is the path the ego has already driven. The **GnBu** trace is the plan the model is currently working from, which stays on screen until the next re-plan replaces it — that is what makes the receding horizon visible: the trace jumps forward every `planning_interval` frames and then contracts as the ego consumes it.

The closed loop commits its plans into the pose arrays but does not return them, so `collect_ego_plans` re-derives them by asking the same model, at the same planning frames, for the ego's plan on the simulated state. Over the window in which a plan is authoritative — from the planning frame up to the next re-plan — that reproduces the committed trajectory to within a metre. It also requires the rebuilt participants to carry a speed: the model reads `state.speed` as the initial speed, and a state without velocity would plan every agent as if it were standing still.

The camera follows the ego, and `MatplotlibRenderer` draws in world coordinates, so `auto_scale` is switched off and the view window is re-centred on the ego every frame; letting it auto-scale would fit the entire map into the axes and shrink the interaction to a few pixels.

`PLAYBACK_STEP = 1` renders every simulated step, so the animation plays at 1x: 90 frames at `interval=100` ms give a 9-second video for the 9-second scenario, the same convention the BITS and LimSim demos use. Set it to 2 to halve the frame count and play at 2x.

In [ ]:
def rebuild_render_participants(poses, participants, step_ms):
    """Turn the runner's pose arrays into participants the camera can render.

    The pose arrays carry positions and headings only, so the speed is recovered
    from neighbouring poses. The behavior model reads ``state.speed`` as the
    agent's initial speed, and a state without velocity would silently plan
    every agent as if it were standing still.
    """
    rebuilt = {}
    dt = step_ms / 1000.0
    for agent_id, array in poses.items():
        valid = [index for index, row in enumerate(array) if row[0] != -1.0]
        if not valid:
            continue
        source = participants.get(agent_id)
        trajectory = Trajectory(id_=agent_id, fps=round(1000.0 / step_ms, 3), stable_freq=True)
        last = None
        for index in range(valid[0], valid[-1] + 1):
            row = array[index]
            if row[0] != -1.0:
                last = (float(row[0]), float(row[1]), float(row[3]))
            speed = 0.0
            after = min(index + 1, valid[-1])
            before = max(index - 1, valid[0])
            if array[after][0] != -1.0 and array[before][0] != -1.0 and after > before:
                distance = np.hypot(
                    array[after][0] - array[before][0], array[after][1] - array[before][1]
                )
                speed = float(distance / (dt * (after - before)))
            trajectory.add_state(
                State(frame=index * step_ms, x=last[0], y=last[1], heading=last[2], speed=speed)
            )
        rebuilt[agent_id] = Vehicle(
            agent_id,
            "vehicle",
            trajectory=trajectory,
            length=float(getattr(source, "length", 4.8)),
            width=float(getattr(source, "width", 1.9)),
        )
        # The closed loop hands every agent its last ground-truth position as a
        # routing goal; mirror it so the re-derived plan follows the same lanes.
        if source is not None and source.trajectory.last_state is not None:
            rebuilt[agent_id].goal_xy = (
                source.trajectory.last_state.x,
                source.trajectory.last_state.y,
            )
    return rebuilt


def collect_ego_plans(model, render_participants, map_, config, ego_id, end_index):
    """Re-derive the ego's planned trajectory at every planning frame.

    The runner commits its plans into the pose arrays but does not return them,
    so the plan trace is recovered by asking the same model, at the same frames,
    for the ego's plan on the simulated state. Over the window in which a plan
    is authoritative (up to the next replan) this reproduces what the runner
    committed.
    """
    plans = {}
    for index in range(config.planning_warmup_steps, end_index + 1, config.planning_interval):
        frame = index * config.step_ms
        result = model.plan(render_participants, map_, frame, agent_ids=[ego_id])
        trajectory = result.trajectories.get(ego_id)
        if trajectory is None:
            continue
        plans[frame] = [
            (f, trajectory.get_state(f).x, trajectory.get_state(f).y) for f in trajectory.frames
        ]
    return plans


def render_closed_loop_animation(
    render_participants, map_, playback_frames, ego_id, plans=None, resolution=(1200, 800)
):
    for roadline in map_.roadlines.values():
        if roadline.type_ is None:
            roadline.type_ = "roadline"
    camera = BEVCamera(id_=0, map_=map_, perception_range=PERCEPTION_RANGE)
    prev_road, prev_part = set(), set()
    renderer = MatplotlibRenderer(
        xlim=(-VIEW_WIDTH / 2, VIEW_WIDTH / 2),
        ylim=(-VIEW_HEIGHT / 2, VIEW_HEIGHT / 2),
        resolution=resolution,
        auto_scale=False,
    )
    renderer.enable_trajectory_gradient()
    ego_trajectory = render_participants[ego_id].trajectory
    plan_frames = sorted(plans) if plans else []

    def update(frame):
        nonlocal prev_road, prev_part
        renderer._remove_trajectory_lines()
        pids = [pid for pid, p in render_participants.items() if frame in p.trajectory.history_states]
        if ego_trajectory.has_state(frame):
            ego_pose = ego_trajectory.get_state(frame)
            cam_pos = Point(ego_pose.x, ego_pose.y)
            renderer.ax.set_xlim(ego_pose.x - VIEW_WIDTH / 2, ego_pose.x + VIEW_WIDTH / 2)
            renderer.ax.set_ylim(ego_pose.y - VIEW_HEIGHT / 2, ego_pose.y + VIEW_HEIGHT / 2)
        else:
            ego_pose = None
            cam_pos = Point(0.0, 0.0)
        geometry_data, prev_road, prev_part = camera.update(
            frame, render_participants, pids, prev_road, prev_part, cam_pos
        )
        renderer.update(geometry_data)
        if ego_pose is not None:
            past = [f for f in playback_frames if f <= frame and ego_trajectory.has_state(f)]
            if len(past) >= 2:
                points = [
                    (ego_trajectory.get_state(f).x, ego_trajectory.get_state(f).y) for f in past
                ]
                renderer.draw_gradient_trace(list(reversed(points)), cm.BuPu)
            if plan_frames:
                issued = [f for f in plan_frames if f <= frame]
                if issued:
                    # Only the part of the plan that has not been driven yet, so
                    # the trace always reads forward from the ego.
                    ahead = [(x, y) for f, x, y in plans[issued[-1]] if f > frame]
                    if len(ahead) >= 2:
                        renderer.draw_gradient_trace([(ego_pose.x, ego_pose.y)] + ahead, cm.GnBu)
        renderer.ax.set_title(
            f"InterSim closed loop: {ego_id}  |  frame {frame}  |  active: {len(pids)}", fontsize=7
        )

    return FuncAnimation(renderer.fig, update, frames=playback_frames, interval=100, repeat=True)

In [ ]:
def run_interactive_scenario(
    parser,
    file_name=None,
    folder=None,
    map_path=None,
    map_config=None,
    ego_id=None,
    config=None,
    frame_ms0=0,
    name="scenario",
    resolution=(1200, 800),
    **parse_kwargs,
):
    """Parse one scenario, run the InterSim closed loop and return its animation."""
    config = config or INTERSIM_CFG

    print("Parsing scenario ...")
    participants, time_range = parser.parse_trajectory(
        file=file_name, folder=folder, **parse_kwargs
    )

    map_ = None
    if hasattr(parser, "parse_map"):
        map_ = parser.parse_map(file=file_name, folder=folder, **parse_kwargs)
    if map_ is None and map_path is not None:
        print(f"  loading map from {map_path}")
        map_ = OSMParser(lanelet2=True).parse(file_path=map_path, configs=map_config)
    print(f"  participants: {len(participants)},  frames: {time_range},  lanes: {len(map_.lanes)}")

    if ego_id is None:
        ego_id = choose_ego(participants)
    ego = participants[ego_id]
    ego.color = EGO_COLOR

    model = InterSimBehaviorModel(config)
    plan_frame = frame_at_or_after(ego, WARMUP_MS)
    result = plan_interaction_scene(model, participants, map_, plan_frame, ego_id)
    ego_edges = [e for e in result.relations if ego_id in (e.influencer, e.reactor)]
    print(f"  ego: {ego_id}  |  planning frame: {plan_frame}  |  modelled agents: "
          f"{len(result.scene_agent_ids)}  |  directed relations: {len(result.relations)} "
          f"({len(ego_edges)} involving the ego)")
    for edge in ego_edges:
        role = "ego yields to" if edge.reactor == ego_id else "ego passes before"
        print("    %s -> %s   (reactor step %s, frame diff %s)  [%s %s]"
              % (edge.influencer, edge.reactor, edge.reactor_step, edge.frame_diff, role,
                 edge.influencer if edge.reactor == ego_id else edge.reactor))
    if len(result.relations) > len(ego_edges):
        print("    ... and %d more edges among neighbouring vehicles"
              % (len(result.relations) - len(ego_edges)))
    print("  ego action:", result.actions.get(ego_id))

    plot_directed_relations(
        participants,
        map_,
        result,
        ego_id,
        plan_frame,
        title="InterSim directed relations - %s (frame %s ms)" % (name, plan_frame),
        save_to=Path("../../tests/runtime") / ("intersim_relations_%s.png" % name),
    )

    rolling = model.run_closed_loop(participants, map_, ego_id=ego_id, frame_ms0=frame_ms0)
    print("  closed loop: collided=%s  front/side/rear=%d/%d/%d  offroad=%d  progress=%.2f m  "
          "controlled=%d" % (rolling.collided, rolling.front_collisions, rolling.side_collisions,
                             rolling.rear_collisions, rolling.offroad_scenarios, rolling.progress,
                             rolling.total_agents_controlled))

    render_participants = rebuild_render_participants(rolling.poses, participants, config.step_ms)
    # The rebuilt participants are fresh Vehicle objects, so the ego colour has to
    # be re-applied to the object the renderer actually receives.
    render_participants[ego_id].color = EGO_COLOR
    playback_frames = [
        index * config.step_ms for index in range(0, rolling.end_index + 1, PLAYBACK_STEP)
    ]
    plans = collect_ego_plans(model, render_participants, map_, config, ego_id, rolling.end_index)
    print(f"  rendering {len(playback_frames)} frames, {len(plans)} re-derived ego plans ...")
    return render_closed_loop_animation(
        render_participants, map_, playback_frames, ego_id, plans=plans, resolution=resolution
    )

### Example 1: WOMD — validation_interactive, Scenario 2

This example runs the InterSim closed loop on an official WOMD interactive validation scenario. The ego vehicle approaches a large junction while several vehicles cross its path, so the directed relation graph contains both directions: one vehicle is the influencer for the ego, and the ego is the influencer for two others.

That is also the first thing worth noticing about the printed output — a relation is not the same as a braking action. The ego is the reactor of `6412 -> 1`, yet its action is still `follow`: the resolver only marks an agent `"yield"` when its baseline actually has to be braked, and here the ego is slow enough to clear the conflict on its own. The relation graph tells you who *would* have to give way; `actions` tells you who actually had to slow down.

In the animation below the ego vehicle is shown in pink and its driven path is drawn with a gradient trace; every other vehicle keeps its ground-truth motion unless the closed loop pulled it into the ego's relevant set.

Adjust the folder below to match your local data layout.

In [ ]:
ani_womd_2 = run_interactive_scenario(
    WOMDParser(),
    file_name="validation_interactive.tfrecord-00000-of-00150",
    folder="../../../data/womd/uncompressed/validation_interactive",
    scenario_id=2,
    name="womd_2",
)
ani_womd_2

### Example 2: WOMD — validation_interactive, Scenario 9

A second WOMD interactive scenario, picked because the ego is genuinely on the yielding side. It is almost stationary as the scenario opens (`0.06 m/s` at the planning frame), the relation graph contains `2608 -> 2478` with the ego as the reactor, and the printed action for the ego is `yield` — one of the few scenarios probed where a relation turns into an actual braking action for the ego rather than staying a no-op obligation.

In the animation the ego stays put while the conflict resolves and then accelerates to roughly 12 m/s, so the gradient trace behind it starts late and then stretches out. This scenario is reused in Example 4, where the same run is repeated with the other relation modes.

In [ ]:
ani_womd_9 = run_interactive_scenario(
    WOMDParser(),
    file_name="validation_interactive.tfrecord-00000-of-00150",
    folder="../../../data/womd/uncompressed/validation_interactive",
    scenario_id=9,
    name="womd_9",
)
ani_womd_9

### Example 3: inD (LevelX) — Location 1, Recording 07

This example demonstrates InterSim on the **inD** dataset using `LevelXParser("inD")`, which processes the data at **25 Hz**. The road network is loaded from a Lanelet2 `.osm` map using `OSMParser` together with `IND_MAP_CONFIG`. Recordings **00–06** correspond to **Location 1** (`inD_1`), **07–17** to **Location 2**, and the remaining recordings to the other inD locations.

In this scenario the ego vehicle navigates an intersection where surrounding vehicles create ambiguous interaction patterns, which is exactly the setting the directed relation mechanism was designed for. Note how the LevelX example takes the same arguments as the WOMD ones — only the parser and the map source differ, because the behavior model itself only ever sees `participants` and a `Map`.

!!! info "Highway datasets"

    A HighD example is deliberately omitted. Its ~30 m/s highway speeds mean vehicles stay far apart relative to the 40 m `interaction_distance`, so no pair shares a conflict inside the planning horizon and the relation graph comes back empty. The closed loop still runs and the ego still follows its lane, but there is no interaction to show.

Adjust the paths below to match your local data layout.

In [ ]:
ani_ind = run_interactive_scenario(
    LevelXParser("inD"),
    file_name=7,
    folder="../../../data/LevelX/inD/data",
    map_path="../../data/inD_map/inD_1.osm",
    map_config=IND_MAP_CONFIG["inD_1"],
    name="ind_7",
)
ani_ind

### Example 4: Relation Modes — Directed vs All-Yield

InterSim's behavioral claim is that the *direction* of the relation matters. `relation_mode` selects how the direction is resolved:

| Mode | Meaning |
|------|---------|
| `"directed"` | the later arrival at the conflict point is the reactor (the reproduction default) |
| `"directed_tie"` | like `directed`, but simultaneous crossings yield on both sides |
| `"yield_all"` | both sides of every imminent conflict brake — the all-yield reference |
| `"nn"` | the learned M2I direction arbiter replaces the geometric direction on vehicle-vehicle pairs |

The cell below replays the scenario from Example 2 once per mode and reports the upstream-aligned metrics. No animation is produced here; the point is the metric table. `directed` lets the ego keep its right of way and make progress, while `directed_tie` and `yield_all` brake both sides of every conflict, so the ego yields to traffic it would otherwise pass first and its progress drops accordingly.

!!! warning

    `"nn"` needs the M2I relation checkpoint and is therefore not exercised in this tutorial. The remaining modes are closed-loop safe and need no weights.

In [ ]:
parser = WOMDParser()
comparison_participants, _ = parser.parse_trajectory(
    9, file="validation_interactive.tfrecord-00000-of-00150", folder="../../../data/womd/uncompressed/validation_interactive"
)
comparison_map = parser.parse_map(
    9, file="validation_interactive.tfrecord-00000-of-00150", folder="../../../data/womd/uncompressed/validation_interactive"
)
comparison_ego = choose_ego(comparison_participants)

print("%-14s %8s %8s %8s %8s %10s %10s" % (
    "relation_mode", "front", "side", "rear", "offroad", "progress", "controlled"))
for relation_mode in ("directed", "directed_tie", "yield_all"):
    cfg = InterSimConfig(cruise_speed=8.0, relation_mode=relation_mode)
    outcome = InterSimBehaviorModel(cfg).run_closed_loop(
        comparison_participants, comparison_map, ego_id=comparison_ego
    )
    print("%-14s %8d %8d %8d %8d %10.2f %10d" % (
        relation_mode, outcome.front_collisions, outcome.side_collisions,
        outcome.rear_collisions, outcome.offroad_scenarios, outcome.progress,
        outcome.total_agents_controlled))

## Quick Configurations

The table below offers three preset configurations for common use cases. InterSim's cost is dominated by parsing and by the number of re-planning frames, not by any network inference, so the presets mostly trade metrics fidelity against wall-clock time.

| Use Case | Key Settings |
|----------|-------------|
| **Fast preview** | `horizon_steps=30`, `planning_interval=20`, `interaction_distance=20`, `cruise_speed=8` |
| **Reproduction** (current) | `horizon_steps=80`, `planning_interval=10`, `interaction_distance=40`, `cruise_speed=8` |
| **High-fidelity eval** | `horizon_steps=80`, `planning_interval=5`, `interaction_distance=60`, `cruise_speed=8` |

!!! warning

    Two settings have to be respected or the closed loop silently stops being meaningful:

    - `cruise_speed` **must** be overridden. Its default is `70.0`, and the baseline profile accelerates a free vehicle toward `max(v0, cruise_speed)`, so a scenario run with the default would be simulated at 70 m/s. The reproduction uses `cruise_speed=8.0`.
    - `use_relation_model` and `use_marginal_model` must stay `False`. Both raise `NotImplementedError` in the model constructor, because the learned predictors are not ported.

    `relation_mode`, `interaction_distance`, `planning_interval` and the yield margins can be changed freely.

## What Stays Internal

This tutorial deliberately stops at the public behavior-model surface. Internals such as the lane matching helpers, the conflict resolver, the pose-level relation scanner, and the reproduction metrics harness are not part of the user path.

Two capabilities of the port are configured but not exercised here:

- the learned relation direction (`relation_mode="nn"`), which needs the M2I relation checkpoint and replaces the geometric direction on vehicle-vehicle pairs;
- the marginal predictor, which is closed in this port (`use_marginal_model` must stay `False`).

Both belong to the reproduction studies rather than to this quickstart. Changing the parser, the file name and the ego heuristic is enough to replay any other scenario in any dataset Tactics2D supports with the exact same code.